# 🔀 Train a Sparse Crosscoder (Lindsey 2024)

**What is a crosscoder?** A single sparse dictionary that reads *and* writes multiple residual‑stream layers at once. Instead of training one SAE per layer (and then wondering whether feature `f_3721` at layer 11 is the same concept as feature `g_9017` at layer 31), a crosscoder gives you **one feature space shared across layers**. This is what you want for circuits and model diffing.

### Primary sources

- Lindsey, Templeton, Marcus, Conerly, Batson, Olah (Anthropic), *Sparse Crosscoders for Cross‑Layer Features and Model Diffing*, Oct 2024 — <https://transformer-circuits.pub/2024/crosscoders/index.html>
- Follow‑up: *Crosscoder diffing update*, 2025 — <https://transformer-circuits.pub/2025/crosscoder-diffing-update/index.html>
- Distinct from Dunefsky **transcoders** (arXiv:2406.11944) and Anthropic **CLTs** (attribution‑graphs methods, 2025).
- Community reference impl: ckkissane/crosscoder-model-diff-replication. `SAELens` ships Transcoder but **not** CrossCoder yet — so this notebook is greenfield.

### Why we care (our stack)

Our `qwen36-27b-sae-papergrade` repo has three independent SAEs at L11/L31/L55. Those features live in *three different bases* — there is no principled way to say “L11 feature 42 equals L31 feature 99.” A crosscoder **ties them**: one dictionary, one feature index, with per‑layer decoder heads. You can then classify every feature as *early‑only / persistent / late‑only* (Lindsey §3) and build real cross‑layer circuits.

### Algorithm

```
layers = [6, 12, 18]          # 3 residual sites
L = len(layers)
D = d_model                    # 2304 for Gemma‑2 2B

x = concat([resid[l] for l in layers], dim=-1)   # (B, L*D)
pre = x @ W_enc + b_enc                          # (B, N)
z   = topk(ReLU(pre), k)                         # sparse (B, N)
x_hat = z @ W_dec + b_dec                        # (B, L*D)

loss = sum_l MSE(x_hat[:,lD:(l+1)D], x[:,lD:(l+1)D])
     + λ * sum_f sum_l || W_dec[f, l, :] ||_2       # per-feature per-layer L2
```

Per‑layer L2 regularization (not Frobenius on the whole decoder) is the Lindsey trick: it encourages features to concentrate on the layers where they actually fire.

### This notebook

Demo scale: Gemma‑2 2B, 3 layers, N=8192 features, K=64 TopK, 20M token budget. Runs in ~20–30 min on a T4. Drop in `Qwen/Qwen3.6-27B-A3B` + layers `[11, 31, 55]` + N=65k + 100M tokens for paper‑grade (\~1–2 h on an A100).

In [ ]:
!pip -q install --upgrade "transformers>=4.44" "accelerate>=0.34" "safetensors>=0.4" "huggingface_hub>=0.25" "datasets>=2.20" einops tqdm matplotlib

## 1. Configuration

Hyper‑parameters live in one place. Edit `BASE_MODEL`, `LAYERS`, `D_MODEL` to scale up to Qwen3.6‑27B.

In [ ]:
import os, math, json, time, hashlib
from pathlib import Path

# --- base model (demo: Gemma-2 2B; paper-grade: Qwen/Qwen3.6-27B-A3B) ---
BASE_MODEL   = 'google/gemma-2-2b'
LAYERS       = [6, 12, 18]        # three residual sites
D_MODEL      = 2304               # Gemma-2 2B hidden size (5120 for Qwen3.6-27B)

# --- crosscoder dictionary ---
N_FEATURES   = 8192               # smaller than typical per-layer SAE (e.g. 32k) -> real compression across layers
K_TOPK       = 64                 # activations per token, summed across features
LAMBDA_REG   = 1e-3               # per-feature per-layer L2 weight (Lindsey §2)

# --- data / optimization ---
TOKEN_BUDGET = 20_000_000         # 20M tokens (demo). Paper-grade: 100M+
SEQ_LEN      = 512
FWD_BATCH    = 4                  # base-model forward batch (sequences)
BATCH_SIZE   = 2048               # crosscoder training batch (tokens)
LR           = 2e-4
WARMUP       = 1000
CHECKPOINT_EVERY_TOKENS = 2_000_000

# --- publishing ---
HF_USERNAME  = os.environ.get('HF_USERNAME', 'caiovicentino1')
HF_REPO      = f'{HF_USERNAME}/gemma2-2b-crosscoder-L6-L12-L18'
LOCAL_OUT    = Path('/content/crosscoder_out')
LOCAL_OUT.mkdir(parents=True, exist_ok=True)

print(f'Base model     : {BASE_MODEL}')
print(f'Layers         : {LAYERS}  (L={len(LAYERS)})')
print(f'D_model        : {D_MODEL}   -> input width L*D = {len(LAYERS)*D_MODEL}')
print(f'N features     : {N_FEATURES}')
print(f'K (TopK)       : {K_TOPK}')
print(f'Token budget   : {TOKEN_BUDGET:,}')
print(f'HF repo        : {HF_REPO}')

## 2. Authenticate & load base model

Base model in **bf16 + SDPA**, frozen. We do not touch its weights.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login, HfApi, create_repo

HF_TOKEN = os.environ.get('HF_TOKEN') or (
    __import__('getpass').getpass('HF token (write scope): ') if 'google.colab' in str(type(__builtins__)) or True else None
)
if HF_TOKEN:
    login(HF_TOKEN, add_to_git_credential=False)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
assert device == 'cuda', 'Crosscoder training wants a GPU.'

print(f'Loading {BASE_MODEL} ...')
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=torch.bfloat16,          # transformers 5.x: dtype= (NOT torch_dtype=)
    attn_implementation='sdpa',
    device_map={'': device},
)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

# pick the residual-carrying transformer block list
if hasattr(model, 'model') and hasattr(model.model, 'layers'):
    blocks = model.model.layers
elif hasattr(model, 'model') and hasattr(model.model, 'language_model'):
    blocks = model.model.language_model.layers
else:
    raise RuntimeError('Could not locate layer list on this arch — inspect state_dict keys.')
print(f'Transformer has {len(blocks)} blocks. Hooking {LAYERS}.')

## 3. Triple‑layer activation streamer

One forward pass, three hooks, one stacked tensor. We stream from The Pile / C4 style text so the crosscoder sees real pretraining distribution.

Shape contract: for every forward of `(FWD_BATCH, SEQ_LEN)` we emit

- an `(FWD_BATCH * SEQ_LEN, L * D_MODEL)` float32 tensor, rows shuffled, chunked into `BATCH_SIZE`-row mini‑batches.

In [ ]:
from datasets import load_dataset

def text_stream():
    """Infinite generator of raw text strings."""
    ds = load_dataset('allenai/c4', 'en', split='train', streaming=True)
    for row in ds:
        t = row.get('text', '')
        if t and len(t) > 200:
            yield t

class TripleLayerHook:
    """Captures residual-stream output at every layer in LAYERS during one fwd pass."""
    def __init__(self, blocks, layers):
        self.layers = layers
        self.bufs = {l: None for l in layers}
        self.hs = []
        for l in layers:
            h = blocks[l].register_forward_hook(self._make(l))
            self.hs.append(h)
    def _make(self, l):
        def hook(_mod, _inp, out):
            # transformer block output: tuple(hidden, ...) or tensor
            h = out[0] if isinstance(out, tuple) else out
            self.bufs[l] = h.detach()
        return hook
    def pop_stack(self):
        # stack on a new first dim -> (L, B, T, D)
        stacked = torch.stack([self.bufs[l] for l in self.layers], dim=0)
        for l in self.layers:
            self.bufs[l] = None
        return stacked
    def close(self):
        for h in self.hs:
            h.remove()

def activation_stream(model, tok, blocks, layers, d_model,
                     seq_len=SEQ_LEN, fwd_batch=FWD_BATCH, batch_size=BATCH_SIZE):
    """Yields float32 tensors of shape (BATCH_SIZE, L*D_MODEL) forever."""
    hooker = TripleLayerHook(blocks, layers)
    L = len(layers)
    text_iter = text_stream()
    pending = []  # list of (N, L*D) float32 CPU tensors
    try:
        while True:
            # gather fwd_batch sequences of seq_len tokens
            batch_texts = []
            while len(batch_texts) < fwd_batch:
                batch_texts.append(next(text_iter))
            enc = tok(batch_texts, return_tensors='pt', max_length=seq_len,
                      truncation=True, padding='max_length')
            ids = enc['input_ids'].to(device)
            with torch.no_grad():
                model(ids)
            stacked = hooker.pop_stack()          # (L, B, T, D) bf16
            # -> (B*T, L, D) -> (B*T, L*D) float32
            stacked = stacked.permute(1, 2, 0, 3).contiguous()  # (B, T, L, D)
            flat = stacked.reshape(-1, L * d_model).float()     # (B*T, L*D)
            pending.append(flat.cpu())
            rows = sum(p.shape[0] for p in pending)
            while rows >= batch_size:
                pool = torch.cat(pending, dim=0)
                perm = torch.randperm(pool.shape[0])
                pool = pool[perm]
                out = pool[:batch_size]
                leftover = pool[batch_size:]
                pending = [leftover] if leftover.shape[0] else []
                rows = leftover.shape[0]
                yield out.to(device, non_blocking=True)
    finally:
        hooker.close()

# smoke test: one mini-batch
_stream = activation_stream(model, tok, blocks, LAYERS, D_MODEL)
_b = next(_stream)
print(f'Sanity batch shape: {tuple(_b.shape)}  dtype={_b.dtype}  device={_b.device}')
assert _b.shape == (BATCH_SIZE, len(LAYERS) * D_MODEL)
del _b, _stream
torch.cuda.empty_cache()

## 4. CrossCoder module

Single encoder `W_enc: (L*D, N)`, single decoder `W_dec: (N, L*D)` that we logically view as `(N, L, D)` per‑layer heads. TopK sparsity, per‑feature per‑layer decoder renorm, geometric‑median init for `b_dec`.

In [ ]:
class CrossCoder(nn.Module):
    def __init__(self, n_features: int, n_layers: int, d_model: int, k_topk: int):
        super().__init__()
        self.N = n_features
        self.L = n_layers
        self.D = d_model
        self.K = k_topk
        in_dim = n_layers * d_model

        # fp32 params (base model stays bf16)
        # Xavier scaled for L*D fan-in
        self.W_enc = nn.Parameter(
            torch.empty(in_dim, n_features, dtype=torch.float32)
        )
        self.W_dec = nn.Parameter(
            torch.empty(n_features, in_dim, dtype=torch.float32)
        )
        self.b_enc = nn.Parameter(torch.zeros(n_features, dtype=torch.float32))
        self.b_dec = nn.Parameter(torch.zeros(in_dim, dtype=torch.float32))

        nn.init.xavier_uniform_(self.W_enc, gain=1.0 / math.sqrt(n_layers))
        # decoder: small random, then renorm per layer
        nn.init.normal_(self.W_dec, std=1.0 / math.sqrt(n_features))
        with torch.no_grad():
            self.renorm_decoder_()

    @torch.no_grad()
    def renorm_decoder_(self):
        """Unit L2 norm per (feature, layer) slice. Lindsey §2 standard."""
        W = self.W_dec.data.view(self.N, self.L, self.D)
        norms = W.norm(dim=-1, keepdim=True).clamp_min(1e-8)
        W.div_(norms)

    @torch.no_grad()
    def geometric_median_init_b_dec_(self, x_samples: torch.Tensor, iters: int = 64):
        """Weiszfeld on a batch of activations to init b_dec."""
        y = x_samples.mean(dim=0).float()
        for _ in range(iters):
            d = (x_samples - y).norm(dim=-1).clamp_min(1e-6)
            w = 1.0 / d
            y = (w[:, None] * x_samples).sum(0) / w.sum()
        self.b_dec.data.copy_(y)

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, L*D) fp32 -> z: (B, N) sparse TopK."""
        pre = (x - self.b_dec) @ self.W_enc + self.b_enc
        acts = F.relu(pre)
        # TopK along feature dim
        vals, idx = acts.topk(self.K, dim=-1)
        z = torch.zeros_like(acts)
        z.scatter_(-1, idx, vals)
        return z

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        return z @ self.W_dec + self.b_dec

    def forward(self, x: torch.Tensor):
        z = self.encode(x)
        x_hat = self.decode(z)
        return x_hat, z

    def per_layer_decoder_norms(self) -> torch.Tensor:
        """Returns (N, L) tensor of ||W_dec[f, l, :]||_2 — the feature→layer map."""
        W = self.W_dec.detach().view(self.N, self.L, self.D)
        return W.norm(dim=-1).cpu()

cc = CrossCoder(N_FEATURES, len(LAYERS), D_MODEL, K_TOPK).to(device)

# geometric-median init for b_dec from a fresh activation batch
_stream = activation_stream(model, tok, blocks, LAYERS, D_MODEL)
with torch.no_grad():
    _probe = next(_stream).float()
    cc.geometric_median_init_b_dec_(_probe, iters=64)
del _probe, _stream
torch.cuda.empty_cache()

n_params = sum(p.numel() for p in cc.parameters())
print(f'Crosscoder params: {n_params/1e6:.2f} M   (fp32)')
print(f'  W_enc: {tuple(cc.W_enc.shape)}')
print(f'  W_dec: {tuple(cc.W_dec.shape)}  (view as ({N_FEATURES}, {len(LAYERS)}, {D_MODEL}))')

## 5. Training loop

Standard Adam + cosine schedule with linear warmup. Loss is **per‑layer reconstruction MSE** (so no layer can get starved) plus Lindsey‑style **per‑feature per‑layer L2** regularization. We log `var_expl_L{l}` for each layer separately — a single dictionary covering 3 layers will have per‑layer variance‑explained slightly below dedicated SAEs; that's the expected cost of cross‑layer coherence.

In [ ]:
from tqdm.auto import tqdm

opt = torch.optim.Adam(cc.parameters(), lr=LR, betas=(0.9, 0.999))

def lr_at(step, total_steps, warmup=WARMUP, base=LR):
    if step < warmup:
        return base * (step + 1) / warmup
    prog = (step - warmup) / max(1, total_steps - warmup)
    return base * 0.5 * (1 + math.cos(math.pi * prog))

total_steps = TOKEN_BUDGET // BATCH_SIZE
print(f'Training for {total_steps:,} steps ({TOKEN_BUDGET:,} tokens, batch {BATCH_SIZE}).')

stream = activation_stream(model, tok, blocks, LAYERS, D_MODEL)
L = len(LAYERS)

log = {'step': [], 'loss': [], 'recon': [], 'reg': [], **{f'var_expl_L{l}': [] for l in LAYERS}}
tokens_seen = 0
next_ckpt = CHECKPOINT_EVERY_TOKENS
t0 = time.time()

pbar = tqdm(range(total_steps), dynamic_ncols=True)
for step in pbar:
    x = next(stream)                                   # (B, L*D)
    lr_now = lr_at(step, total_steps)
    for g in opt.param_groups:
        g['lr'] = lr_now

    x_hat, z = cc(x)
    # per-layer MSE -> sum
    x_parts     = x.view(-1, L, D_MODEL)
    x_hat_parts = x_hat.view(-1, L, D_MODEL)
    mse_per_layer = ((x_hat_parts - x_parts) ** 2).mean(dim=(0, 2))   # (L,)
    recon = mse_per_layer.sum()

    # per-feature per-layer L2 regularization on decoder
    W = cc.W_dec.view(cc.N, L, D_MODEL)
    reg = W.norm(dim=-1).sum() / cc.N                  # mean-over-features, sum-over-layers

    loss = recon + LAMBDA_REG * reg

    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
    with torch.no_grad():
        cc.renorm_decoder_()

    tokens_seen += BATCH_SIZE

    if step % 25 == 0:
        with torch.no_grad():
            var_tot = x_parts.var(dim=(0, 2)).clamp_min(1e-8)             # (L,)
            var_res = (x_hat_parts - x_parts).var(dim=(0, 2))             # (L,)
            var_expl = (1.0 - var_res / var_tot).tolist()
        log['step'].append(step)
        log['loss'].append(float(loss.item()))
        log['recon'].append(float(recon.item()))
        log['reg'].append(float(reg.item()))
        for li, l in enumerate(LAYERS):
            log[f'var_expl_L{l}'].append(var_expl[li])
        pbar.set_postfix({
            'loss': f'{loss.item():.3f}',
            **{f'VE_L{l}': f'{var_expl[i]:.3f}' for i, l in enumerate(LAYERS)},
            'tok': f'{tokens_seen/1e6:.1f}M',
            'lr':   f'{lr_now:.1e}',
        })

    # crash-safe checkpoint
    if tokens_seen >= next_ckpt:
        ckpt_path = LOCAL_OUT / f'crosscoder_step{step:06d}.safetensors'
        from safetensors.torch import save_file
        save_file({
            'W_enc': cc.W_enc.detach().cpu(),
            'W_dec': cc.W_dec.detach().cpu(),
            'b_enc': cc.b_enc.detach().cpu(),
            'b_dec': cc.b_dec.detach().cpu(),
        }, str(ckpt_path))
        (LOCAL_OUT / 'train_log.json').write_text(json.dumps(log))
        print(f'\n  ckpt saved: {ckpt_path.name}  ({tokens_seen/1e6:.1f}M tok)')
        next_ckpt += CHECKPOINT_EVERY_TOKENS

print(f'\nDone in {(time.time()-t0)/60:.1f} min. Final loss={log["loss"][-1]:.4f}')

## 6. Final checkpoint + upload

Writes `crosscoder_final.safetensors` + `cfg.json`. The config records `architecture="crosscoder"` so downstream tooling can distinguish this from a per‑layer SAE.

In [ ]:
from safetensors.torch import save_file

final_path = LOCAL_OUT / 'crosscoder_final.safetensors'
save_file({
    'W_enc': cc.W_enc.detach().cpu(),
    'W_dec': cc.W_dec.detach().cpu(),
    'b_enc': cc.b_enc.detach().cpu(),
    'b_dec': cc.b_dec.detach().cpu(),
}, str(final_path))

cfg = {
    'architecture': 'crosscoder',
    'base_model': BASE_MODEL,
    'layers': LAYERS,
    'd_model': D_MODEL,
    'n_features': N_FEATURES,
    'k_topk': K_TOPK,
    'lambda_reg': LAMBDA_REG,
    'token_budget': TOKEN_BUDGET,
    'seq_len': SEQ_LEN,
    'batch_size': BATCH_SIZE,
    'lr': LR,
    'warmup': WARMUP,
    'activation': 'topk_relu',
    'decoder_renorm': 'per_feature_per_layer_unit_l2',
    'regularization': 'per_feature_per_layer_l2_sum',
    'reference': 'Lindsey et al. 2024, transformer-circuits.pub/2024/crosscoders',
}
(LOCAL_OUT / 'cfg.json').write_text(json.dumps(cfg, indent=2))
(LOCAL_OUT / 'train_log.json').write_text(json.dumps(log))
print(f'Wrote {final_path.name} ({final_path.stat().st_size/1e6:.1f} MB)')
print(f'Wrote cfg.json:')
print(json.dumps(cfg, indent=2))

api = HfApi()
try:
    create_repo(HF_REPO, exist_ok=True, private=False, token=HF_TOKEN)
    api.upload_folder(
        folder_path=str(LOCAL_OUT),
        repo_id=HF_REPO,
        repo_type='model',
        token=HF_TOKEN,
        ignore_patterns=['*.ipynb_checkpoints*'],
    )
    print(f'\nUploaded to https://huggingface.co/{HF_REPO}')
except Exception as e:
    print(f'HF upload skipped / failed: {e}')

## 7. Cross‑layer feature analysis (the Lindsey §3 killer use case)

For every feature we have three scalars: `||W_dec[f, L6]||`, `||W_dec[f, L12]||`, `||W_dec[f, L18]||`. Since we unit‑normalize per (feature, layer), these norms are all ≈ 1 **when a feature is active at that layer**. Using the encoder frequency (how often feature `f` fires into each layer's portion) would be the Lindsey‑exact metric; for the decoder‑norm‑recipe we instead look at **effective‑norm‑weighted activations** — we compute the actual per‑layer contribution `|z_f| * ||W_dec[f, l, :]||` on a validation batch, which is the right proxy for “how much does feature f matter at layer l.”

We then label every feature:

- **persistent**: normalized contribution ≥ 0.3 at all three layers
- **early‑only**: concentrates on L6 only
- **mid‑only**: concentrates on L12 only
- **late‑only**: concentrates on L18 only
- **mixed**: two of three layers dominant

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# collect per-layer feature contribution on a fresh validation stream
N_VAL_BATCHES = 32
val_stream = activation_stream(model, tok, blocks, LAYERS, D_MODEL)
L = len(LAYERS)

contrib = torch.zeros(N_FEATURES, L, device=device)   # accumulated |z_f| * ||W_dec[f,l,:]||
freq    = torch.zeros(N_FEATURES, device=device)      # fire count per feature
W_view  = cc.W_dec.detach().view(N_FEATURES, L, D_MODEL)
layer_norms = W_view.norm(dim=-1)                     # (N, L)

cc.eval()
with torch.no_grad():
    for _ in tqdm(range(N_VAL_BATCHES), desc='feature analysis'):
        x = next(val_stream)
        z = cc.encode(x)                              # (B, N)
        active = (z > 0).float()
        freq += active.sum(0)
        # contribution per feature per layer: mean |z_f| * decoder-norm at that layer
        mean_abs_z = z.abs().mean(0)                  # (N,)
        contrib += mean_abs_z[:, None] * layer_norms  # broadcast (N,1) * (N,L)

contrib = contrib.cpu()
freq    = freq.cpu()

# normalize per feature (row sums to 1 for features that fired at all)
row_sum = contrib.sum(dim=1, keepdim=True).clamp_min(1e-8)
frac    = (contrib / row_sum).numpy()                  # (N, L)

THRESH_PERSIST = 0.30
THRESH_DOMINANT = 0.70

labels = []
dead_count = 0
for f in range(N_FEATURES):
    if freq[f].item() == 0:
        labels.append('dead')
        dead_count += 1
        continue
    r = frac[f]
    if (r >= THRESH_PERSIST).all():
        labels.append('persistent')
    elif r[0] >= THRESH_DOMINANT:
        labels.append(f'early_L{LAYERS[0]}_only')
    elif r[1] >= THRESH_DOMINANT:
        labels.append(f'mid_L{LAYERS[1]}_only')
    elif r[2] >= THRESH_DOMINANT:
        labels.append(f'late_L{LAYERS[2]}_only')
    else:
        labels.append('mixed')

from collections import Counter
counter = Counter(labels)
total_live = N_FEATURES - dead_count
print(f'Total features     : {N_FEATURES}')
print(f'Dead features      : {dead_count} ({100*dead_count/N_FEATURES:.1f}%)')
print(f'Live features      : {total_live}')
print('\nFeature-type distribution (over live features):')
for k, v in counter.most_common():
    pct = 100*v/total_live if k != 'dead' and total_live else 0
    print(f'  {k:24s} {v:6d}   ({pct:5.1f}%)')

# plot
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
names  = [k for k in counter.keys() if k != 'dead']
counts = [counter[k] for k in names]
axes[0].bar(names, counts)
axes[0].set_title('Crosscoder feature types')
axes[0].set_ylabel('# features')
axes[0].tick_params(axis='x', rotation=30)
# ternary-ish scatter: early vs late contribution fraction
mask = np.array([l != 'dead' for l in labels])
axes[1].scatter(frac[mask, 0], frac[mask, -1], s=3, alpha=0.4)
axes[1].set_xlabel(f'frac @ L{LAYERS[0]}')
axes[1].set_ylabel(f'frac @ L{LAYERS[-1]}')
axes[1].set_title('Feature layer-placement (early vs late)')
axes[1].plot([0,1],[0,1],'k--',lw=0.5)
plt.tight_layout()
fig_path = LOCAL_OUT / 'feature_map.png'
plt.savefig(fig_path, dpi=130)
plt.show()
print(f'Saved {fig_path}')

In [ ]:
# Save per-feature per-layer decoder-norm + contribution + label map and push
feat_map = {
    'layers': LAYERS,
    'n_features': N_FEATURES,
    'thresholds': {'persistent': THRESH_PERSIST, 'dominant': THRESH_DOMINANT},
    'per_feature': [
        {
            'feature': int(f),
            'decoder_norm_per_layer':   [float(layer_norms[f, li].item()) for li in range(L)],
            'contribution_per_layer':   [float(contrib[f, li].item())     for li in range(L)],
            'contribution_frac':        [float(frac[f, li])               for li in range(L)],
            'firing_frequency':         float(freq[f].item()),
            'label':                    labels[f],
        }
        for f in range(N_FEATURES)
    ],
    'summary': dict(counter),
}
fm_path = LOCAL_OUT / 'crosscoder_feature_map.json'
fm_path.write_text(json.dumps(feat_map))
print(f'Wrote {fm_path.name} ({fm_path.stat().st_size/1e6:.2f} MB)')

try:
    api.upload_file(
        path_or_fileobj=str(fm_path),
        path_in_repo='crosscoder_feature_map.json',
        repo_id=HF_REPO,
        repo_type='model',
        token=HF_TOKEN,
    )
    api.upload_file(
        path_or_fileobj=str(LOCAL_OUT / 'feature_map.png'),
        path_in_repo='feature_map.png',
        repo_id=HF_REPO,
        repo_type='model',
        token=HF_TOKEN,
    )
    print(f'Pushed feature map + figure to https://huggingface.co/{HF_REPO}')
except Exception as e:
    print(f'HF upload skipped / failed: {e}')

print('\nDone. This is a Lindsey-2024-style sparse crosscoder: one dictionary,')
print(f'shared across layers {LAYERS}, {N_FEATURES} features, TopK={K_TOPK}.')